# ⚡ Módulo 11 - Notebook 02: Arquitectura Spark y DataFrames

## 🏗️ Particiones, Esquemas y Tipos de Datos Distribuidos

**Libro:** Saliendo de lo Pandito  
**Módulo:** 11 - PySpark Core SparkSession  
**Duración estimada:** 70 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Entender** particiones y paralelismo  
✅ **Definir** esquemas explícitos con StructType  
✅ **Manejar** tipos de datos de Spark  
✅ **Optimizar** número de particiones  
✅ **Crear** DataFrames con esquema estricto

---

## 📋 Pre-requisitos

* ✅ Notebook 11_01 completado (Introducción a Spark)
* ✅ Conocimiento de SparkSession
* ✅ Familiaridad con tipos de datos

---

## 📚 Contenido

1. Particiones y Paralelismo
2. StructType y StructField
3. Tipos de Datos en Spark
4. Esquemas Explícitos vs Inferidos
5. Optimización de Particiones
6. Caso Integrador: DataFrame Empresarial con Esquema

---

## 💡 Por qué importa

**Esquemas y particiones son fundamentales:**

* 🏗️ **Esquemas:** Definen estructura y tipos
* ⚡ **Particiones:** Control del paralelismo
* 🎯 **Performance:** Optimización de ejecución
* 🛡️ **Calidad:** Validación de datos

**El corazón del procesamiento distribuido**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df_spark = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df_spark.count():,}")
    print(f"   🏛️ Particiones: {df_spark.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📋 Esquema del DataFrame:")
    df_spark.printSchema()
    
    print(f"\n🔍 Información de particiones:")
    print(f"   • Cada partición se procesa en paralelo")
    print(f"   • Spark divide los datos automáticamente")
    print(f"   • Número óptimo depende del cluster")
    
    print(f"\n🎯 Este notebook explorará:")
    print(f"   • Esquemas (StructType)")
    print(f"   • Tipos de datos de Spark")
    print(f"   • Optimización de particiones")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_spark = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Arquitectura Interna: Particiones y Esquemas

### 🗂️ Particiones: Divide y Vencerás

**Partición:** Un pedazo del DataFrame distribuido en un Executor.

**Concepto:**
```
DataFrame con 1000 filas, 4 particiones:

Partición 1 (Executor 1): Filas 1-250
Partición 2 (Executor 2): Filas 251-500
Partición 3 (Executor 3): Filas 501-750
Partición 4 (Executor 4): Filas 751-1000
```

**Ventaja:** Las 4 particiones se procesan **en paralelo**.

---

### ⚡ Paralelismo y Particiones

**Regla de oro:**
```
Número de particiones ≈ 2-3 × número de cores
```

**Ejemplo:**
* Cluster con 8 cores → 16-24 particiones óptimas
* Cluster con 32 cores → 64-96 particiones óptimas

**Ver particiones:**
```python
df.rdd.getNumPartitions()  # Consultar
```

**Cambiar particiones:**
```python
df_repartitioned = df.repartition(16)  # Más particiones
df_coalesced = df.coalesce(4)  # Menos particiones (sin shuffle)
```

---

### 📐 StructType: Esquemas Explícitos

**StructType** define la estructura de un DataFrame.

**Sintaxis:**
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("nombre", StringType(), nullable=False),
    StructField("edad", IntegerType(), nullable=True),
    StructField("ciudad", StringType(), nullable=True)
])

data = [("Juan", 30, "Mendoza"), ("María", 25, "Buenos Aires")]
df = spark.createDataFrame(data, schema=schema)
```

**Ventajas:**
* ✅ **Performance:** No necesita inferir tipos
* ✅ **Validación:** Rechaza datos inválidos
* ✅ **Documentación:** Esquema explícito

---

### 🔤 Tipos de Datos en Spark

**Tipos principales:**

| Spark Type | Python | SQL | Ejemplo |
|------------|--------|-----|----------|
| **StringType** | str | STRING | "Hola" |
| **IntegerType** | int | INT | 42 |
| **LongType** | int | BIGINT | 9999999999 |
| **FloatType** | float | FLOAT | 3.14 |
| **DoubleType** | float | DOUBLE | 3.141592 |
| **BooleanType** | bool | BOOLEAN | True |
| **DateType** | date | DATE | 2024-01-01 |
| **TimestampType** | datetime | TIMESTAMP | 2024-01-01 10:30:00 |
| **DecimalType** | Decimal | DECIMAL(10,2) | 1234.56 |

**Importar:**
```python
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    DateType, TimestampType
)
```

---

### 🔍 Esquema Inferido vs Explícito

**Inferido (automático):**
```python
df = spark.read.csv("file.csv", header=True, inferSchema=True)
# Spark adivina los tipos
```

**Explícito (recomendado en producción):**
```python
schema = StructType([
    StructField("fecha", DateType()),
    StructField("ventas", DoubleType())
])
df = spark.read.csv("file.csv", header=True, schema=schema)
# Tipos garantizados
```

**Comparación:**

| Aspecto | Inferido | Explícito |
|---------|----------|----------|
| **Velocidad** | Lento (escanea datos) | Rápido |
| **Precisión** | Puede fallar | 100% preciso |
| **Validación** | No | Sí |
| **Uso** | Exploración | Producción |

---

### 🔧 Operaciones con Particiones

**1️⃣ Repartition (shuffle completo):**
```python
# Redistribuir datos (costoso)
df_new = df.repartition(16)
```

**2️⃣ Coalesce (reducir sin shuffle):**
```python
# Reducir particiones (barato)
df_new = df.coalesce(4)
```

**3️⃣ Repartition por columna:**
```python
# Agrupar por columna (útil para joins)
df_new = df.repartition("sucursal_id")
```

**Regla:**
* Aumentar particiones → `repartition()`
* Reducir particiones → `coalesce()`

---

### 💼 Caso de Uso: DataFrame Empresarial

**Problema:** Cargar ventas con esquema estricto.

```python
from pyspark.sql.types import *

# Definir esquema
schema = StructType([
    StructField("fecha", DateType(), nullable=False),
    StructField("sucursal_id", IntegerType(), nullable=False),
    StructField("producto", StringType(), nullable=False),
    StructField("ventas", DecimalType(10, 2), nullable=False),
    StructField("costo", DecimalType(10, 2), nullable=True)
])

# Cargar con esquema
df = spark.read.csv("/path/ventas.csv", schema=schema, header=True)

# Validación automática: rechaza filas inválidas
```

**Ventaja:** Errores detectados en carga, no en análisis.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, LongType,
    DateType, TimestampType, BooleanType, DecimalType
)
from pyspark.sql.functions import col
import warnings
warnings.filterwarnings('ignore')

print("🏗️ ARQUITECTURA SPARK: Particiones y Esquemas")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Particiones y paralelismo")
print("  • StructType y StructField")
print("  • Tipos de datos de Spark")
print("  • Esquemas explícitos vs inferidos")

print("\n📖 Métodos clave:")
print("  - df.rdd.getNumPartitions()  # Ver particiones")
print("  - df.repartition(n)  # Cambiar particiones")
print("  - StructType([StructField(...)])  # Definir esquema")
print("  - df.printSchema()  # Ver esquema")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')